<a href="https://colab.research.google.com/github/PolinaBoeva/predictive_models_dota2/blob/models/experiments/Boeva/%D1%87%D0%B5%D0%BA%D0%BF%D0%BE%D0%B8%D0%BD%D1%82_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

In [ ]:
y_train = pd.read_csv('y_train.csv', index_col=0)
X_train = pd.read_csv('X_train.csv', index_col=0)
X_test = pd.read_csv('X_test.csv', index_col=0)
y_test = pd.read_csv('y_test.csv', index_col=0)

In [ ]:
y_train_heroes = pd.read_csv('y_train_heroes.csv', index_col=0)
X_train_heroes = pd.read_csv('X_train_heroes.csv', index_col=0)
X_test_heroes = pd.read_csv('X_test_heroes.csv', index_col=0)
y_test_heroes = pd.read_csv('y_test_heroes.csv', index_col=0)

In [ ]:
metrics_df = pd.DataFrame(columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"])

In [ ]:
def get_model_metrics(y_test, predictions, X_test,  model_name, predict_proba=None):
    global metrics_df
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    if predict_proba:
        roc_auc = roc_auc_score(y_test, predict_proba(X_test)[:, 1])  # Для AUC-ROC используем predict_proba
    else:
        roc_auc = "N/A"
    print(f"Metrics for {model_name}:")
    print(f"Accuracy: {round(accuracy, 3)}")
    print(f"Precision: {round(precision, 3)}")
    print(f"Recall: {round(recall, 3)}")
    print(f"F1 Score: {round(f1, 3)}")
    print(f"ROC-AUC: {round(roc_auc, 3)}")

    new_metrics = pd.DataFrame([[model_name, round(accuracy, 3), round(precision, 3),
                                 round(recall, 3), round(f1, 3), round(roc_auc, 3)]],
                                columns=metrics_df.columns)

    metrics_df = pd.concat([metrics_df, new_metrics], ignore_index=True)


# Модели логистической регрессии

In [ ]:
# Бейзлайн модель. Увеличила количество интераций для сходимости
pipeline = Pipeline([
    ('scaler_', StandardScaler()),
    ('model_', LogisticRegression(max_iter=500))
])

# Обучение пайплайна
pipeline.fit(X_train, y_train)

# Прогнозирование на тестовой выборке
predictions = pipeline.predict(X_test)
predict_proba = pipeline.predict_proba

# Оценка модели
baseline_metrics = get_model_metrics(y_test, predictions, X_test, 'baseline', predict_proba)
baseline_metrics

Metrics for baseline:
Accuracy: 0.591
Precision: 0.587
Recall: 0.659
F1 Score: 0.621
ROC-AUC: 0.632


In [ ]:
# Подбор гиперпараметров для логистической регрессии
param_distributions = {
    'model__C': [0.01, 1, 0.1, 100],
    'model__penalty': ['l1', 'none'],
    'model__solver': ['liblinear', 'saga'],
    'model__max_iter': [100, 500, 5000],
    'model__class_weight': [None, 'balanced']
}

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

f1_scorer = make_scorer(f1_score)

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions,
    n_iter=10,
    cv=3,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=0,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best parameters for Logistic Regression:", random_search.best_params_)
print("Best F1 score:", random_search.best_score_)

best_C = random_search.best_params_['model__C']
best_penalty = random_search.best_params_['model__penalty']
best_class_weight = random_search.best_params_['model__class_weight']

Best parameters for Logistic Regression: {'model__solver': 'liblinear', 'model__penalty': 'l1', 'model__max_iter': 100, 'model__class_weight': None, 'model__C': 0.01}
Best F1 score: 0.59927289376972


In [ ]:
best_model = random_search.best_estimator_
predictions = best_model.predict(X_test)
predict_proba = best_model.predict_proba
logistic_metrics = get_model_metrics(y_test, predictions, X_test, 'logistic', predict_proba)
logistic_metrics

Metrics for logistic:
Accuracy: 0.586
Precision: 0.585
Recall: 0.635
F1 Score: 0.609
ROC-AUC: 0.631


In [ ]:
X_train_mean = X_train.loc[:, X_train.columns.str.contains('mean|radiant_win')]
X_test_mean = X_test.loc[:, X_test.columns.str.contains('mean|radiant_win')]

In [ ]:
# Модель на средних
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())])

param_distributions = {
    'model__C': [0.01, 1, 0.1, 100],
    'model__penalty': ['l1', 'none'],
    'model__solver': ['liblinear', 'saga'],
    'model__max_iter': [100, 500, 5000],
    'model__class_weight': [None, 'balanced']
}

f1_scorer = make_scorer(f1_score)

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions,
    n_iter=10,
    cv=3,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=0,
    random_state=42
)

random_search.fit(X_train_mean, y_train)

print(f"Лучшие гиперпараметры: {random_search.best_params_}")

Лучшие гиперпараметры: {'model__solver': 'saga', 'model__penalty': 'l1', 'model__max_iter': 100, 'model__class_weight': None, 'model__C': 100}


In [ ]:
best_model = random_search.best_estimator_
predictions = best_model.predict(X_test_mean)
predict_proba = best_model.predict_proba
logistic_mean_metrics = get_model_metrics(y_test, predictions, X_test_mean, 'logistic_mean', predict_proba)
logistic_mean_metrics

Metrics for logistic_mean:
Accuracy: 0.597
Precision: 0.591
Recall: 0.671
F1 Score: 0.629
ROC-AUC: 0.643


Модель только на средних превосходит модель на всех данных по всем метрикам. Это означает, что модель на средних имеет более высокую точность, полноту и сбалансированную метрику F1, что делает ее более эффективной в задаче предсказания победы команды Radiant.

# Другие модели





In [ ]:
# SVM
from sklearn.svm import SVC

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(probability=True, class_weight='balanced'))
])

pipeline.fit(X_train, y_train)

prediction = pipeline.predict(X_test)
predict_proba = pipeline.predict_proba

In [ ]:
svm_metrics = get_model_metrics(y_test, predictions, X_test, 'svm', predict_proba)
svm_metrics

Metrics for svm:
Accuracy: 0.597
Precision: 0.591
Recall: 0.671
F1 Score: 0.629
ROC-AUC: 0.631


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# Подбор гиперпараметров LDA
param_distributions = {
    'model__solver': ['lsqr', 'eigen', 'svd'],
    'model__priors': [None, 'uniform', [0.3, 0.7], [0.7, 0.3]],
    'model__n_components': np.arange(1, 4),
}

lda_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearDiscriminantAnalysis())
])

f1_scorer = make_scorer(f1_score)

random_search_lda = RandomizedSearchCV(lda_pipeline, param_distributions, n_iter=10, cv=5, scoring=f1_scorer, random_state=42)
random_search_lda.fit(X_train, y_train)

print("Best parameters for LDA:", random_search_lda.best_params_)

Best parameters for LDA: {'model__solver': 'svd', 'model__priors': [0.3, 0.7], 'model__n_components': 1}


In [ ]:
best_model = random_search_lda.best_estimator_
predictions = best_model.predict(X_test)
predict_proba = best_model.predict_proba
LDA_metrics = get_model_metrics(y_test, predictions, X_test, 'LDA', predict_proba)
LDA_metrics

Metrics for LDA:
Accuracy: 0.527
Precision: 0.519
Recall: 0.986
F1 Score: 0.68
ROC-AUC: 0.633


In [ ]:
# Наивный байесовский классификатор
from sklearn.naive_bayes import GaussianNB

nb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GaussianNB())
])

nb_pipeline.fit(X_train, y_train)
nb_predictions = nb_pipeline.predict(X_test)
predict_proba = nb_pipeline.predict_proba
nb_metrics = get_model_metrics(y_test, nb_predictions, X_test, 'Naive Bayes', predict_proba)
nb_metrics

Metrics for Naive Bayes:
Accuracy: 0.555
Precision: 0.612
Recall: 0.343
F1 Score: 0.44
ROC-AUC: 0.598


In [ ]:
metrics_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,baseline,0.591,0.587,0.659,0.621,0.632
1,logistic,0.586,0.585,0.635,0.609,0.631
2,logistic_mean,0.597,0.591,0.671,0.629,0.643
3,svm,0.597,0.591,0.671,0.629,0.631
4,LDA,0.527,0.519,0.986,0.680,0.633
5,Naive Bayes,0.555,0.612,0.343,0.440,0.598


Логистическая регрессия и её модификации, а также LDA, показывают схожие результаты по значениям accuracy, precision, recall и F1 score. Хотя наивный байесовский классификатор показывает хорошую precision, его recall слишком низкий.
Для повышения качества предсказаний, можно попробовать использовать более сложные модели.

In [ ]:
# Дерево решений
from sklearn.tree import DecisionTreeClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier())
])

pipeline.fit(X_train, y_train)

predictions = pipeline.predict(X_test)
predict_proba = pipeline.predict_proba
DT_metrics = get_model_metrics(y_test, predictions, X_test, 'Decision', predict_proba)
DT_metrics

Metrics for Decision:
Accuracy: 0.526
Precision: 0.534
Recall: 0.537
F1 Score: 0.536
ROC-AUC: 0.526


In [ ]:
# Дерево решений
from sklearn.tree import DecisionTreeClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier(class_weight='balanced'))
])

pipeline.fit(X_train, y_train)

predictions = pipeline.predict(X_test)
predict_proba = pipeline.predict_proba
DT_metrics = get_model_metrics(y_test, predictions, X_test, 'Decision', predict_proba)
DT_metrics

Metrics for Decision:
Accuracy: 0.529
Precision: 0.538
Recall: 0.528
F1 Score: 0.533
ROC-AUC: 0.529


Дерево решений в данном случае показывает относительно слабые результаты, модель не очень хорошо отделяет положительный класс от отрицательного, о чем свидетельствует низкий ROC-AUC.

In [ ]:
# Случайный лес
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(class_weight='balanced'))
])

pipeline.fit(X_train, y_train)

predictions = pipeline.predict(X_test)
predict_proba = pipeline.predict_proba

RT_metrics = get_model_metrics(y_test, predictions, X_test, 'Random Forest', predict_proba)
RT_metrics

Metrics for Random Forest:
Accuracy: 0.581
Precision: 0.586
Recall: 0.602
F1 Score: 0.594
ROC-AUC: 0.615


Random Forest показывает значительное улучшение по сравнению с Decision Tree. Модель достигает лучших результатов по всем метрикам, включая accuracy, precision, recall, F1 score и ROC-AUC.

In [ ]:
pip install catboost

In [ ]:
# catboost
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train, y_train)

In [ ]:
predictions = pipeline.predict(X_test)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test, predictions, X_test, 'CatBoost', predict_proba)
catboost_metrics

Metrics for CatBoost:
Accuracy: 0.599
Precision: 0.605
Recall: 0.613
F1 Score: 0.609
ROC-AUC: 0.647


CatBoost показывает достаточно хорошие результаты по большинству метрик, особенно по recall и F1 Score, что говорит о том, что модель эффективно находит положительные примеры.

In [ ]:
metrics_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,baseline,0.591,0.587,0.659,0.621,0.632
1,logistic,0.586,0.585,0.635,0.609,0.631
2,logistic_mean,0.597,0.591,0.671,0.629,0.643
3,svm,0.597,0.591,0.671,0.629,0.631
4,LDA,0.527,0.519,0.986,0.680,0.633
5,Naive Bayes,0.555,0.612,0.343,0.440,0.598
6,Decision,0.526,0.534,0.537,0.536,0.526
7,Decision,0.529,0.538,0.528,0.533,0.529
8,Random Forest,0.581,0.586,0.602,0.594,0.615
9,CatBoost,0.599,0.605,0.613,0.609,0.647


CatBoost показал лучшие результаты среди моделей с Accuracy = 0.599, Precision = 0.605, Recall = 0.613, F1 = 0.609 и ROC-AUC = 0.647, что делает его наиболее сбалансированным вариантом. Logistic Mean и SVM также демонстрируют достойные показатели. LDA имеет высокий Recall, но низкую Accuracy. Naive Bayes сильно теряет в Recall, а Decision Trees уступают по всем метрикам.

Для дальнейшего улучшения возможно:

• Подбор гиперпараметров моделей CatBoost и Random Forest.

• Использование других моделей.

• Feature Engineering (добавление информации по героям и тд.)


# Feature Engineering

In [ ]:
# Посчитаем разницу по значениям для команд team_1 team_2
def calculate_team_differences(df):
    """
    Вычисляет разницу между колонками team_1 и team_2 в DataFrame.

    :param df: DataFrame с колонками, содержащими "team_1" и "team_2"
    :return: Новый DataFrame с разницами значений
    """
    diff_columns = {}

    for col in df.columns:
        if "team_1" in col:
            col_team_2 = col.replace("team_1", "team_2")
            if col_team_2 in df.columns:
                new_col_name = col.replace("_team_1", "_diff")
                diff_columns[new_col_name] = df[col] - df[col_team_2]

    return pd.DataFrame(diff_columns)

In [ ]:
X_train_diff = calculate_team_differences(X_train)
X_test_diff = calculate_team_differences(X_test)

In [ ]:
# catboost только на колонках с разницей
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train_diff, y_train)

In [ ]:
# катбуст показывает хорошие метрики даже только на колонках с разницей
predictions = pipeline.predict(X_test_diff)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test, predictions, X_test_diff, 'CatBoost diff', predict_proba)
catboost_metrics

Metrics for CatBoost diff:
Accuracy: 0.596
Precision: 0.599
Recall: 0.622
F1 Score: 0.61
ROC-AUC: 0.638


In [ ]:
# Добавим новые переменные к изначальному датасету
X_train_merge = pd.concat([X_train, X_train_diff], axis=1)
X_test_merge = pd.concat([X_test, X_test_diff], axis=1)

In [ ]:
# catboost
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train_merge, y_train)

In [ ]:
# Удалось повысить качество предсказания относительно модели без новых колонок
predictions = pipeline.predict(X_test_merge)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test, predictions, X_test_merge, 'CatBoost merge', predict_proba)
catboost_metrics

Metrics for CatBoost merge:
Accuracy: 0.592
Precision: 0.597
Recall: 0.614
F1 Score: 0.605
ROC-AUC: 0.649


# Оптимизация гиперпараметров

In [ ]:
!pip install optuna

In [ ]:
import optuna
from catboost import CatBoostClassifier

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-5, 0.3),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-5, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'one_hot_max_size': trial.suggest_int('one_hot_max_size', 2, 10),
        'random_state': 42
    }

    model = CatBoostClassifier(**params, verbose=0)

    model.fit(X_train_merge, y_train)

    y_pred_proba = model.predict_proba(X_test_merge)[:, 1]

    roc_auc = roc_auc_score(y_test, y_pred_proba)

    return roc_auc

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"Лучшие параметры: {study.best_params}")
print(f"Лучший ROC AUC: {study.best_value}")

[I 2025-03-13 11:00:12,836] A new study created in memory with name: no-name-149411d8-5180-4fed-9b8c-eda87af8f718
[I 2025-03-13 11:01:38,293] Trial 0 finished with value: 0.6307323376727909 and parameters: {'iterations': 770, 'depth': 6, 'learning_rate': 0.0001902602200270678, 'l2_leaf_reg': 0.01731516593658743, 'border_count': 201, 'one_hot_max_size': 4}. Best is trial 0 with value: 0.6307323376727909.
[I 2025-03-13 11:02:51,776] Trial 1 finished with value: 0.634906299306569 and parameters: {'iterations': 472, 'depth': 7, 'learning_rate': 0.002146019450775934, 'l2_leaf_reg': 1.0534208458531729, 'border_count': 188, 'one_hot_max_size': 2}. Best is trial 1 with value: 0.634906299306569.
[I 2025-03-13 11:09:06,903] Trial 2 finished with value: 0.6316747054373791 and parameters: {'iterations': 434, 'depth': 10, 'learning_rate': 0.03004936793327931, 'l2_leaf_reg': 0.00021641605283192743, 'border_count': 213, 'one_hot_max_size': 9}. Best is trial 1 with value: 0.634906299306569.
[I 2025-03

Лучшие параметры: {'iterations': 468, 'depth': 6, 'learning_rate': 0.03828694027547689, 'l2_leaf_reg': 1.7598081363931177, 'border_count': 207, 'one_hot_max_size': 2}
Лучший ROC AUC: 0.6467783266055377


In [ ]:
best_params = study.best_params
best_model = CatBoostClassifier(**best_params)
best_model.fit(X_train_merge, y_train)

# Оценка на тестовой выборке
y_pred_proba_best = best_model.predict_proba(X_test_merge)[:, 1]  # Получаем вероятности для положительного класса
best_roc_auc = roc_auc_score(y_test, y_pred_proba_best)
print(f"ROC AUC с лучшими гиперпараметрами: {best_roc_auc}")

In [ ]:
best_params

{'iterations': 468,
 'depth': 6,
 'learning_rate': 0.03828694027547689,
 'l2_leaf_reg': 1.7598081363931177,
 'border_count': 207,
 'one_hot_max_size': 2}

In [ ]:
predictions = best_model.predict(X_test_merge)
y_pred_proba_best = best_model.predict_proba
catboost_metrics = get_model_metrics(y_test, predictions, X_test_merge, 'CatBoost optuna', y_pred_proba_best)
catboost_metrics

Metrics for CatBoost optuna:
Accuracy: 0.592
Precision: 0.592
Recall: 0.638
F1 Score: 0.614
ROC-AUC: 0.645


Подбор гиперпараметров не улучшил качество модели, метрика Accuracy отсалась прежней, f1 увеличился, ROC-AUC снизился

# lightgbm

In [ ]:
!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

pipeline = Pipeline([
    ('model', LGBMClassifier(random_state=42))
])

pipeline.fit(X_train_merge, y_train)

In [ ]:
predictions = pipeline.predict(X_test_merge)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test, predictions, X_test_merge, 'LGBMClassifier', predict_proba)
catboost_metrics

Metrics for LGBMClassifier:
Accuracy: 0.589
Precision: 0.592
Recall: 0.618
F1 Score: 0.605
ROC-AUC: 0.64


# Методы снижения размерности

In [ ]:
# Снижение размерности PCA
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=270)
X_reduced = svd.fit_transform(X_train_merge)

explained_variance = np.cumsum(svd.explained_variance_ratio_)

n_components = np.argmax(explained_variance >= 0.95) + 1

print(f"Достаточно оставить {n_components} компонент для 95% дисперсии.")

Достаточно оставить 9 компонент для 95% дисперсии.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_merge)
X_test_scaled = scaler.transform(X_test_merge)

pca = PCA(n_components=0.99)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Размерности после PCA:")
print("X_train:", X_train_pca.shape)
print("X_test:", X_test_pca.shape)

Размерности после PCA:
X_train: (44330, 109)
X_test: (8042, 109)


In [ ]:
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train_pca, y_train)

In [ ]:
# качество модели не улучшилось
predictions = pipeline.predict(X_test_pca)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test, predictions, X_test_pca, 'CatBoost pca', predict_proba)
catboost_metrics

Metrics for CatBoost pca:
Accuracy: 0.58
Precision: 0.589
Recall: 0.575
F1 Score: 0.582
ROC-AUC: 0.626


In [ ]:
# Снижение размерности LDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_merge)
X_test_scaled = scaler.transform(X_test_merge)
lda = LinearDiscriminantAnalysis(n_components=1)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)

In [ ]:
# catboost
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train_lda, y_train)

In [ ]:
predictions = pipeline.predict(X_test_lda)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test, predictions, X_test_lda, 'CatBoost lda', predict_proba)
catboost_metrics

Metrics for CatBoost lda:
Accuracy: 0.588
Precision: 0.583
Recall: 0.661
F1 Score: 0.62
ROC-AUC: 0.631


# Отбор признаков

In [ ]:
pip install autofeat

In [ ]:
# Отбор на датасете с изначальными признаками
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# feature selection
from autofeat import FeatureSelector
fsel = FeatureSelector(verbose=1)
X_train_selected = fsel.fit_transform(X_train, y_train)
X_test_selected = fsel.transform(X_test)

[featsel] Scaling data...done.


In [ ]:
# catboost
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train_selected, y_train)

In [ ]:
# Качество на уровне лучшей модели с дополнительными признаками
predictions = pipeline.predict(X_test_selected)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test, predictions, X_test_selected, 'CatBoost select', predict_proba)
catboost_metrics

Metrics for CatBoost select:
Accuracy: 0.593
Precision: 0.596
Recall: 0.619
F1 Score: 0.607
ROC-AUC: 0.648


# Рассмотрим данные с признаками героев

In [ ]:
y_train_heroes = pd.read_csv('y_train_heroes.csv', index_col=0)
X_train_heroes = pd.read_csv('X_train_heroes.csv', index_col=0)
X_test_heroes = pd.read_csv('X_test_heroes.csv', index_col=0)
y_test_heroes = pd.read_csv('y_test_heroes.csv', index_col=0)

In [ ]:
X_train_heroes.head()

,previous_kills_avr_heroes_team_1_mean,previous_hero_kills_avr_heroes_team_1_mean,previous_courier_kills_avr_heroes_team_1_mean,previous_observer_kills_avr_heroes_team_1_mean,previous_kills_per_min_avr_heroes_team_1_mean,previous_kda_avr_heroes_team_1_mean,previous_denies_avr_heroes_team_1_mean,previous_hero_healing_avr_heroes_team_1_mean,previous_assists_avr_heroes_team_1_mean,previous_hero_damage_avr_heroes_team_1_mean,...,previous_tower_kills_avr_players_team_2_min,previous_win_avr_players_team_2_mean,previous_win_avr_players_team_2_max,previous_win_avr_players_team_2_min,previous_duration_avr_players_team_2_mean,previous_duration_avr_players_team_2_max,previous_duration_avr_players_team_2_min,previous_first_blood_time_avr_players_team_2_mean,previous_first_blood_time_avr_players_team_2_max,previous_first_blood_time_avr_players_team_2_min
match_id,,,,,,,,,,,,,,,,,,,,,
6947109699,5.500000,5.500000,0.000000,0.750000,0.169753,6.402500,5.500000,1688.000000,17.000000,12758.000000,...,0.0,0.0,0.0,0.0,32.400000,32.400000,32.400000,2.066667,2.066667,2.066667
6947952638,7.833333,7.666667,0.000000,1.666667,0.273104,5.216667,7.500000,2389.000000,14.666667,16532.833333,...,0.0,0.2,1.0,0.0,29.450000,29.450000,29.450000,1.750000,1.750000,1.750000
6948030779,4.708333,4.583333,0.583333,0.583333,0.161977,1.983333,4.875000,53.000000,8.750000,13105.666667,...,0.0,1.0,1.0,1.0,27.416667,27.416667,27.416667,2.516667,2.516667,2.516667
6948105554,3.777778,3.777778,0.666667,0.888889,0.115479,8.942222,4.111111,1919.777778,12.000000,10617.555556,...,0.0,0.5,0.5,0.5,29.958333,29.958333,29.958333,1.258333,1.258333,1.258333
6948219195,6.333333,6.000000,0.000000,0.500000,0.213535,4.648333,8.500000,198.833333,12.833333,15254.333333,...,0.0,1.0,1.0,1.0,29.316667,29.316667,29.316667,4.000000,4.000000,4.000000


In [ ]:
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train_heroes, y_train_heroes)

In [ ]:
# Лучшее качество по ROC-AUC среди всех моделей
predictions = pipeline.predict(X_test_heroes)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test_heroes, predictions, X_test_heroes, 'CatBoost heroes', predict_proba)
catboost_metrics

Metrics for CatBoost heroes:
Accuracy: 0.598
Precision: 0.604
Recall: 0.611
F1 Score: 0.607
ROC-AUC: 0.65


In [ ]:
# переменные с разницей по командам
X_train_diff_heroes = calculate_team_differences(X_train_heroes)
X_test_diff_heroes = calculate_team_differences(X_test_heroes)

In [ ]:
X_train_merge_heroes = pd.concat([X_train_heroes, X_train_diff_heroes], axis=1)
X_test_merge_heroes = pd.concat([X_test_heroes, X_test_diff_heroes], axis=1)

In [ ]:
# catboost
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier())
])

pipeline.fit(X_train_merge_heroes, y_train_heroes)

In [ ]:
# лучшее качество по метрикам ROC-AUC и Accuracy
predictions = pipeline.predict(X_test_merge_heroes)
predict_proba = pipeline.predict_proba

catboost_metrics = get_model_metrics(y_test_heroes, predictions, X_test_merge_heroes, 'CatBoost heroes diff', predict_proba)
catboost_metrics

Metrics for CatBoost heroes diff:
Accuracy: 0.606
Precision: 0.611
Recall: 0.622
F1 Score: 0.616
ROC-AUC: 0.655
